In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [2]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.split_cp import SplitConformalPredictor

Load data

In [3]:
input_points, output_points = load_data("friedman1")

In [4]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)
(
    proper_train_input_points,
    calib_input_points,
    proper_train_output_points,
    calib_output_points,
) = train_test_split(train_input_points, train_output_points, random_state=0)

Instantiate predictor

In [5]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [6]:
predictor = KernelRegression(
    solver="lbfgs", loss_name=loss_name, loss_params=loss_params, lam=0.5
)
predictor.fit(proper_train_input_points, proper_train_output_points)

Instantiate region predictor

In [7]:
conformal_predictor = SplitConformalPredictor(predictor, non_conformity_name="absolute")
conformal_predictor.fit(calib_input_points, calib_output_points)
region_predictor = conformal_predictor.predict(test_input_points)

In [8]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

In [9]:
coverage = np.mean(
    [
        test_output_point in prediction_region
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage)

test coverage:  0.96


In [10]:
prediction_regions

[[array([-1.8859133]),array([1.18261699])],
 [array([-1.52676872]),array([1.54176157])],
 [array([-1.83764914]),array([1.23088115])],
 [array([-1.32814193]),array([1.74038836])],
 [array([-1.25201847]),array([1.81651182])],
 [array([-1.28806188]),array([1.78046841])],
 [array([-1.46598819]),array([1.6025421])],
 [array([-2.1847253]),array([0.88380499])],
 [array([-0.74722237]),array([2.32130792])],
 [array([-1.95948037]),array([1.10904991])],
 [array([-0.98679148]),array([2.08173881])],
 [array([-1.38298277]),array([1.68554752])],
 [array([-1.80694939]),array([1.2615809])],
 [array([-1.75188307]),array([1.31664722])],
 [array([-1.57378783]),array([1.49474246])],
 [array([-0.85293049]),array([2.2155998])],
 [array([-1.5629638]),array([1.50556649])],
 [array([-1.3371265]),array([1.73140379])],
 [array([-1.45781935]),array([1.61071094])],
 [array([-2.00063299]),array([1.0678973])],
 [array([-1.38246173]),array([1.68606856])],
 [array([-1.48521361]),array([1.58331668])],
 [array([-1.606755